# 06 · 用 GPU 加速

深度學習慢，是因為有海量的矩陣運算。**GPU** 能平行處理這些運算，常常快上幾十倍。這堂課教你寫**裝置無關**的程式，並親手量一次 GPU 的加速。

## 學習目標

- 偵測可用裝置（CUDA / Apple MPS / CPU）並寫出裝置無關的程式
- 理解 GPU 為何適合神經網路
- 親手 benchmark CPU vs GPU 的矩陣運算

## 1. 選一個裝置

標準寫法：能用 GPU 就用，否則退回 CPU。Colab 上是 `cuda`，Mac 上是 `mps`。

In [1]:
import torch

if torch.cuda.is_available():
    device = "cuda"           # Colab / NVIDIA GPU
elif torch.backends.mps.is_available():
    device = "mps"            # Apple Silicon
else:
    device = "cpu"
print("使用裝置:", device)

使用裝置: mps


> 在 Colab：**執行階段 → 變更執行階段類型 → 選 GPU（T4）**，這格就會印出 `cuda`，免費。

## 2. 為什麼 GPU 快？

神經網路的核心是**大量矩陣相乘**。CPU 有少數幾個強核心、循序處理；GPU 有數千個小核心，能**同時**算成千上萬個乘加。下面量一次看看——但先講清楚一個誠實的提醒：

In [2]:
import time

def bench(dev, size=4000, reps=20):
    a = torch.randn(size, size, device=dev)
    b = torch.randn(size, size, device=dev)
    if dev != "cpu":
        (a @ b); torch.mps.synchronize() if dev == "mps" else torch.cuda.synchronize()
    t = time.perf_counter()
    for _ in range(reps):
        c = a @ b
    if dev == "mps": torch.mps.synchronize()
    elif dev == "cuda": torch.cuda.synchronize()
    return (time.perf_counter() - t) / reps

cpu_t = bench("cpu")
print(f"CPU:  {cpu_t * 1000:7.1f} ms / matmul")
if device != "cpu":
    gpu_t = bench(device)
    print(f"{device.upper():4}: {gpu_t * 1000:7.1f} ms / matmul")
    print(f"\n{device} 相對 CPU：{cpu_t / gpu_t:.1f}×（>1 才是加速）")
    print("註：Mac 的 MPS 在這種中等大小的『單一』運算上，常因啟動/傳輸成本看不出優勢，")
    print("    甚至比 CPU 慢。真正的加速在 Colab 的 CUDA GPU + 大模型整段訓練時才顯著（常達數十倍）。")

CPU:    191.3 ms / matmul


MPS :   175.2 ms / matmul

mps 相對 CPU：1.1×（>1 才是加速）
註：Mac 的 MPS 在這種中等大小的『單一』運算上，常因啟動/傳輸成本看不出優勢，
    甚至比 CPU 慢。真正的加速在 Colab 的 CUDA GPU + 大模型整段訓練時才顯著（常達數十倍）。


## 3. 裝置無關的訓練

把**模型**和**每個 batch 的資料**都 `.to(device)`，其餘程式碼一字不改。這就是讓同一份程式在 CPU 與 GPU 上都能跑的全部訣竅。

```python
model = MyModel().to(device)
for xb, yb in loader:
    xb, yb = xb.to(device), yb.to(device)   # ← 唯一要加的兩行
    optimizer.zero_grad()
    loss = criterion(model(xb), yb)
    loss.backward()
    optimizer.step()
```

## 小結

- 用 `cuda` / `mps` / `cpu` 的判斷式寫**裝置無關**程式。
- GPU 靠數千核心**平行**做矩陣運算，矩陣越大越快。
- 只要把 model 與每個 batch `.to(device)`，訓練就搬上 GPU 了。

## 練習

1. 把 benchmark 的矩陣 `size` 從 4000 改成 1000 和 8000，加速比怎麼變？
2. 在 Colab 開 GPU，把第 04 課的 CNN 用全部 6 萬筆 MNIST 訓練，比 CPU 快多少？

下一課，學會站在巨人肩膀上——**遷移學習**。